# Notebook for Downloading Events of a Season

### Imports

In [ ]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from sportradar_datacore_api.handball import HandballAPI

In [ ]:
load_dotenv()  # Load environment variables from .env file if present


### Configuration

In [ ]:
NAME_COMPETITION = "1. Handball-Bundesliga"

NAME_SEASON = "DAIKIN HBL 2024/25"
YEAR_SEASON = int(NAME_SEASON.split()[-1].split("/")[0])
YEARS_SEASON = NAME_SEASON.split()[-1].replace("/", "-")

PATH_TO_OUTPUT = os.path.join(
    os.getcwd(), "..", "data", "season_24_25"
)

# create path if it does not exist
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

In [ ]:
import duckdb
con = duckdb.connect(f'../data/mydb{YEARS_SEASON}.duckdb')

### Initialize API

In [ ]:
api = HandballAPI(
    base_url=os.getenv("BASE_URL", ""),
    auth_url=os.getenv("AUTH_URL", ""),
    client_id=os.getenv("CLIENT_ID", ""),
    client_secret=os.getenv("CLIENT_SECRET", ""),
    org_id=os.getenv("CLIENT_ORGANIZATION_ID"),
    scopes=["read:organization"],
    sport="handball",
)

### Get wanted competition ID

In [ ]:
id_competition = api.get_competition_id_by_name(NAME_COMPETITION)

# id_competition = int(id_competition)

# Check if the competition was found
if not id_competition:
    raise ValueError(f"Competition '{NAME_COMPETITION}' not found.")
else:
    print(f"→ Competition '{NAME_COMPETITION}' -> {id_competition}")

id_season = api.get_season_id_by_year(
    competition_id=id_competition, season_year=YEAR_SEASON
)
# Check if the season was found
if not id_season:
    raise ValueError(f"Season '{NAME_SEASON}' not found in competition '{NAME_COMPETITION}'.")
else:
    print(f"→ Season '{NAME_SEASON}' -> {id_season}")

# Get Teams in the Season

In [ ]:
list_entities_season = api.get_teams_by_season_id(season_id=id_season)
# display( pd.json_normalize(list_entities_season[0].to_dict()) )
print(f"→ Number of teams in season '{NAME_SEASON}': {len(list_entities_season)}")

### Cols to keep from get_team_by_id

In [ ]:
columns_to_keep = [
    "entityId",
    # "organizationId",
    "organization",
    # "entityGroupId",
    # "entityGroup",
    # "internationalReference",
    # "status",
    "nameFullLocal",
    # "additionalNames",
    "nameFullLatin",
    "codeLocal",
    "codeLatin",
    # "address",
    # "social",
    # "contacts",
    # "colors",
    # "historicalNames",
    "externalId",
    # "ageGroup",
    # "gender",
    # "standard",
    # "grade",
    # "representing",
    # "discipline",
    # "updated",
    # "added",
    # "defaultVenueId",
    # "alternateVenueIds",
    # "images"
]

In [ ]:
df_teams = pd.DataFrame()

# print len of list_entities_season
print(f"Number of teams in season: {len(list_entities_season)}")

for team in list_entities_season:
    id_team = team.entity_id
    team_details = api.get_team_by_id(entity_id=id_team)
    # print all keys of team_details[0]
    # print("Keys in team_details[0]:")
    # print(json.dumps(list(team_details[0].to_dict().keys()), indent=4, default=str))
    # convert to dataframe
    df_team_details = pd.json_normalize(team_details[0].to_dict())
    # display(entity_details)

    # drop columns that are not in columns list
    df_team_details = df_team_details[[col for col in columns_to_keep if col in df_team_details.columns]]
    # display(df_team_details)
    df_teams = pd.concat([df_teams, df_team_details], ignore_index=True)

    # as json dump
    # print(json.dumps(team_details[0].to_dict(), indent=4, default=str))

In [ ]:
# Drop tables if exist
con.execute("DROP TABLE IF EXISTS teams")
# Create duckdb table
con.execute("""
CREATE TABLE IF NOT EXISTS teams AS SELECT * FROM df_teams
""")

In [ ]:
# plot table
con.execute("SELECT * FROM teams").df()

In [ ]:
columns_to_keep_fixtures = [
    "fixtureId",
    # "organizationId",
    # "organization",
    "seasonId",
    # "season",
    # "practiceDrillType",
    # "internationalReference",
    # "status",
    "fixtureNumber",
    "nameLocal",
    "nameLatin",
    "startTimeLocal",
    "startTimeUTC",
    # "startTimeActualUTC",
    # "endTimeActualUTC",
    # "timesUnconfirmed",
    # "locked",
    # "placingIfWon",
    # "placingIfLost",
    # "attendance",
    # "sellout",
    # "duration",
    # "durationFull",
    # "ticketURL",
    # "stageCode",
    # "stage",
    # "seriesCode",
    # "poolCode",
    # "roundCode",
    # "round",
    "roundNumber",
    # "liveDataAvailable",
    # "liveVideoAvailable",
    # "fixtureType",
    # "maximumPeriodTypeUsed",
    # "competitorType",
    "competitors",
    # "venueId",
    # "venue",
    "externalId",
    # "profileId",
    # "includeInStandings",
    # "updated",
    # "added",
    # "seriesFixtureNumber",
    # "discipline",
    # "broadcasts"
]

columns_to_keep_competitors = [
    "entityId",
    # "conferenceId",
    # "divisionId",
    # "includeInConferenceStatistics",
    "isHome",
    # "includeInRepresentation",
    "draw",
    # "resultStatus",
    "resultPlace",
    # "resultSecondaryScorePlace",
    # "startingNumber",
    "score",
    # "secondaryScore",
    # "shootOutAttempts",
    # "rosterStatus",
    # "isNeutralVenue",
    # "uniformId",
    # "externalId",
]

### Get the fixtures (matches) of a season and insert to duckdb

In [ ]:
list_fixtures = api.get_list_matches_by_season_id(season_id=id_season)

print(f"Found {len(list_fixtures)} fixtures.")

# drop table fixtures if exists
con.execute("DROP TABLE IF EXISTS fixtures")

for match in list_fixtures:
    df_fixture = pd.json_normalize(match.to_dict())
    # drop columns that are not in columns list
    df_fixture = df_fixture[
        [col for col in columns_to_keep_fixtures if col in df_fixture.columns]
    ]

    competitors_expanded = pd.json_normalize(
        df_fixture["competitors"].explode().to_list()
    )
    # drop columns that are not in columns list
    competitors_expanded = competitors_expanded[
        [col for col in columns_to_keep_competitors if col in competitors_expanded.columns]
    ]    
    # insert names to competitors_expanded
    competitors_expanded = competitors_expanded.merge(
        df_teams[["entityId", "nameFullLocal"]],
        left_on="entityId",
        right_on="entityId",
        how="left",
    )
    # display(competitors_expanded)
    # convert competitors_expanded to json and add to df_fixture
    df_fixture = df_fixture.drop(columns=["competitors"])
    df_fixture = df_fixture.assign(competitors= [competitors_expanded.to_dict(orient="records")])
    # insert entityId_home and entityId_away to df_fixture
    df_fixture = df_fixture.assign(
        entityId_home=competitors_expanded[competitors_expanded["isHome"] == True]["entityId"].values[0],
        entityId_away=competitors_expanded[competitors_expanded["isHome"] == False]["entityId"].values[0],
    )
    # insert name_team_home and name_team_away to df_fixture
    df_fixture = df_fixture.assign(
        name_team_home=competitors_expanded[competitors_expanded["isHome"] == True]["nameFullLocal"].values[0],
        name_team_away=competitors_expanded[competitors_expanded["isHome"] == False]["nameFullLocal"].values[0],
    )
    # insert score_home and score_away to df_fixture
    df_fixture = df_fixture.assign(
        score_home=competitors_expanded[competitors_expanded["isHome"] == True]["score"].values[0],
        score_away=competitors_expanded[competitors_expanded["isHome"] == False]["score"].values[0],
    )
    # insert resultPlace_home and resultPlace_away to df_fixture
    df_fixture = df_fixture.assign(
        resultPlace_home=competitors_expanded[competitors_expanded["isHome"] == True]["resultPlace"].values[0],
        resultPlace_away=competitors_expanded[competitors_expanded["isHome"] == False]["resultPlace"].values[0],
    )


    # display(df_fixture)
    # break
    # append to duckdb table
    con.execute(
        """
    CREATE TABLE IF NOT EXISTS fixtures AS SELECT * FROM df_fixture
    """
    )
    con.execute(
        """
    INSERT INTO fixtures SELECT * FROM df_fixture
    """
    )



# plot duplicate fixtureid's
display(
    con.execute(
        """
SELECT fixtureId, COUNT(*) as count FROM fixtures
GROUP BY fixtureId
HAVING count > 1
"""
    ).df()
)

# drop duplicate fixtureid's keeping first
con.execute(
    """
DELETE FROM fixtures
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM fixtures
    GROUP BY fixtureId
)
"""
)
# plot table
display(con.execute("SELECT * FROM fixtures").df())


In [ ]:
df_all_fixtures_in_season = con.execute("SELECT * FROM fixtures").df()

# Drop unused column
if "competitors" in df_all_fixtures_in_season.columns:
    df_all_fixtures_in_season = df_all_fixtures_in_season.drop(columns=["competitors"])

# --- prep & sorting ---
import numpy as np
import pandas as pd

for col in ["score_home", "score_away"]:
    if col in df_all_fixtures_in_season.columns:
        df_all_fixtures_in_season[col] = pd.to_numeric(df_all_fixtures_in_season[col], errors="coerce")

df_all_fixtures_in_season["startTimeUTC"] = pd.to_datetime(
    df_all_fixtures_in_season["startTimeUTC"], errors="coerce"
)

sort_cols = ["startTimeUTC"]
if "fixtureNumber" in df_all_fixtures_in_season.columns:
    sort_cols.append("fixtureNumber")
sort_cols.append("fixtureId")

df_all_fixtures_in_season = df_all_fixtures_in_season.sort_values(sort_cols).reset_index(drop=True)

# --- initialize standings state ---
team_id_cols = ["entityId_home", "entityId_away"]
teams = pd.unique(pd.concat([df_all_fixtures_in_season[c] for c in team_id_cols], ignore_index=True))

name_lookup = {}
if "name_team_home" in df_all_fixtures_in_season.columns and "name_team_away" in df_all_fixtures_in_season.columns:
    home_names = df_all_fixtures_in_season.set_index("entityId_home")["name_team_home"]
    away_names = df_all_fixtures_in_season.set_index("entityId_away")["name_team_away"]
    name_lookup = pd.concat([home_names, away_names]).dropna().groupby(level=0).first().to_dict()

# Add wins/draws/losses + pts_against to the per-team state
standings = {
    tid: {
        "pts": 0,           # points earned by the team (2/1/0)
        "pts_against": 0,   # points opponents earned vs this team
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "name": name_lookup.get(tid, "")
    }
    for tid in teams
}

def points_for_pair(h, a):
    if h > a:
        return 2, 0
    if h < a:
        return 0, 2
    return 1, 1  # draw

def standings_table_df():
    tbl = (
        pd.DataFrame.from_dict(standings, orient="index")
        .assign(gd=lambda x: x["gf"] - x["ga"])
    )
    tbl = tbl.sort_values(
        by=["pts", "gd", "gf", "name"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    tbl["rank"] = range(1, len(tbl) + 1)
    return tbl

# --- iterate fixtures and capture standings after each game ---
standing_after_home, standing_after_away = [], []

# New per-fixture cumulative outputs
wins_home_after, draws_home_after, losses_home_after, pts_against_home_after = [], [], [], []
wins_away_after, draws_away_after, losses_away_after, pts_against_away_after = [], [], [], []

for _, row in df_all_fixtures_in_season.iterrows():
    home_id = row["entityId_home"]
    away_id = row["entityId_away"]
    sh = row["score_home"]
    sa = row["score_away"]

    # if unplayed, just snapshot current ranks & cumulative stats
    if pd.isna(sh) or pd.isna(sa):
        tbl = standings_table_df()
        standing_after_home.append(tbl.loc[home_id, "rank"] if home_id in tbl.index else np.nan)
        standing_after_away.append(tbl.loc[away_id, "rank"] if away_id in tbl.index else np.nan)

        # copy current cumulative values
        wins_home_after.append(standings[home_id]["wins"])
        draws_home_after.append(standings[home_id]["draws"])
        losses_home_after.append(standings[home_id]["losses"])
        pts_against_home_after.append(standings[home_id]["pts_against"])

        wins_away_after.append(standings[away_id]["wins"])
        draws_away_after.append(standings[away_id]["draws"])
        losses_away_after.append(standings[away_id]["losses"])
        pts_against_away_after.append(standings[away_id]["pts_against"])
        continue

    sh, sa = int(sh), int(sa)

    # goals
    standings[home_id]["gf"] += sh
    standings[home_id]["ga"] += sa
    standings[away_id]["gf"] += sa
    standings[away_id]["ga"] += sh

    # points
    p_home, p_away = points_for_pair(sh, sa)
    standings[home_id]["pts"] += p_home
    standings[away_id]["pts"] += p_away

    # wins/draws/losses and pts_against (opponents' points vs this team)
    if sh > sa:  # home win
        standings[home_id]["wins"] += 1
        standings[away_id]["losses"] += 1
        standings[home_id]["pts_against"] += 0
        standings[away_id]["pts_against"] += 2
    elif sh < sa:  # away win
        standings[home_id]["losses"] += 1
        standings[away_id]["wins"] += 1
        standings[home_id]["pts_against"] += 2
        standings[away_id]["pts_against"] += 0
    else:  # draw
        standings[home_id]["draws"] += 1
        standings[away_id]["draws"] += 1
        standings[home_id]["pts_against"] += 1
        standings[away_id]["pts_against"] += 1

    # recompute gd explicitly
    standings[home_id]["gd"] = standings[home_id]["gf"] - standings[home_id]["ga"]
    standings[away_id]["gd"] = standings[away_id]["gf"] - standings[away_id]["ga"]

    # standings after THIS game
    tbl = standings_table_df()
    standing_after_home.append(tbl.loc[home_id, "rank"])
    standing_after_away.append(tbl.loc[away_id, "rank"])

    # push cumulative snapshots for both teams
    wins_home_after.append(standings[home_id]["wins"])
    draws_home_after.append(standings[home_id]["draws"])
    losses_home_after.append(standings[home_id]["losses"])
    pts_against_home_after.append(standings[home_id]["pts_against"])

    wins_away_after.append(standings[away_id]["wins"])
    draws_away_after.append(standings[away_id]["draws"])
    losses_away_after.append(standings[away_id]["losses"])
    pts_against_away_after.append(standings[away_id]["pts_against"])

# attach results to fixtures
df_all_fixtures_in_season["standing_home"] = standing_after_home
df_all_fixtures_in_season["standing_away"] = standing_after_away

df_all_fixtures_in_season["wins_home"] = wins_home_after
df_all_fixtures_in_season["draws_home"] = draws_home_after
df_all_fixtures_in_season["losses_home"] = losses_home_after
df_all_fixtures_in_season["pts_against_home"] = pts_against_home_after

df_all_fixtures_in_season["wins_away"] = wins_away_after
df_all_fixtures_in_season["draws_away"] = draws_away_after
df_all_fixtures_in_season["losses_away"] = losses_away_after
df_all_fixtures_in_season["pts_against_away"] = pts_against_away_after

# final standings (now includes wins/draws/losses/pts_against)
final_table = standings_table_df()
print("Final Standings:")
display(final_table[["rank","name","pts","pts_against","wins","draws","losses","gf","ga","gd"]])

# save to csv
df_all_fixtures_in_season.to_csv("fixtures.csv", index=False)

# save to duckdb
con.execute("DROP TABLE IF EXISTS fixtures_enhanced")
con.execute("CREATE TABLE fixtures_enhanced AS SELECT * FROM df_all_fixtures_in_season")

display((df_all_fixtures_in_season))


### Get fixture

In [ ]:
columns_to_keep_event_list = [
    # "clientId",
    # "clientType",
    "fixtureId",
    # "organizationId",
    # "received",
    # "sport",
    # "topic",
    # "type",
    "class",
    "eventId",
    "eventTime",
    "eventType",
    "subType",
    # "timestamp",
    "attendance",
    # "numberOfPeriods",
    # "periodLength",
    "entityId",
    "personId",
    # "status",
    # "active",
    "bib",
    # "captain",
    "name",
    "position",
    # "starter",
    # "number",
    "scores",
    "periodId",
    # "sequence",
    "playId",
    "clock",
    "success",
    "x",
    "y",
    "attackType",
    "goalKeeperId",
    "location",
    # "options",
    "failureReason",
    # "flagged",
    # "value",
    "emptyNet",
]


In [ ]:
import tqdm

In [ ]:
# query fixtureIds from duckdb
list_fixture_ids = con.execute("SELECT fixtureId FROM fixtures").fetchall()
list_fixture_ids = [fid[0] for fid in list_fixture_ids]

print(f"Found {len(list_fixture_ids)} fixtures.")
df_all_players = pd.DataFrame()
df_all_match_events = pd.DataFrame()

for fid in tqdm.tqdm(list_fixture_ids):
    # print(f"Downloading events for fixtureId: {fid}")

    match_events = api.get_fixture_events_by_id(
        fid, setup_only=False, with_scores=True
    )
    # print(f"  Events count: {len(setup_events)}")

    

    for event in match_events:
        if "data" in event and event["data"]:
            # Merge the data dict into the event dict
            event.update(event["data"])
            del event["data"]  # Remove the original data field
            pass
        if "options" in event and event["options"]:
            # Merge the options dict into the event dict
            event.update(event["options"])
            del event["options"]  # Remove the original options field

    df_match_events = pd.DataFrame(
        match_events, columns=columns_to_keep_event_list
    )
    

    # query teams from duckdb
    df_teams = con.execute("SELECT * FROM teams").df()
    # print(f"df_teams Teams count from duckdb: {len(df_teams)}")
    # display(df_teams)

    # insert namefulllocal to df_match_events from df_teams
    df_match_events = df_match_events.merge(
        df_teams[["entityId", "nameFullLocal"]],
        left_on="entityId",
        right_on="entityId",
        how="left",
    )
    # rename "nameFullLocal" column to "teamName"
    df_match_events = df_match_events.rename(
        columns={"nameFullLocal": "teamName"}
    )
    # filter for class == "setup" and eventType == "person"
    df_setup_events = df_match_events[
        (df_match_events["class"] == "setup")
        & (df_match_events["eventType"] == "person")
    ]
    # print(f"df_match_events Events count: {len(df_match_events)}")
    # display(df_match_events)

    # get list of unique personId's
    unique_person_ids = df_setup_events["personId"].unique()
    list_person_ids = unique_person_ids.tolist()
    # print(f"  Unique personIds count: {len(unique_person_ids)}")
    # get list of unique personId's
    unique_person_names = df_setup_events["name"].unique()
    # print(f"  Unique personNames count: {len(unique_person_names)}")
    # drop duplicates
    df_setup_events = df_setup_events.drop_duplicates(subset=["personId"])
    # print("df_match_events after filtering and drop_duplicates")
    

    
    # display(df_match_events)

    str_person_ids = ",".join(list_person_ids)
    players = api.get_players_by_ids(person_ids=str_person_ids)
    df_players = pd.json_normalize(players)
    # print(f"  Downloaded player details count: {len(df_players)}")
    # display(df_players)
    # insert entityId to df_players from df_match_events

    people_map = (
        df_setup_events[["personId", "entityId", "teamName"]]
        .dropna(subset=["personId"])
        .drop_duplicates(subset=["personId"])
    )

    df_players = df_players.merge(
        people_map, on="personId", how="left", validate="one_to_one"  # ensures no duplication
    )
    # df_players = df_players.merge(
    #     df_match_events[["teamName", "personId", "entityId"]],
    #     left_on="personId",
    #     right_on="personId",
    #     how="left",
    # )
    # print(f"Length of match events before personName insert: {len(df_match_events)}")
    # display(df_match_events)

    # insert nameFullLocal as personName to df_match_events from df_players
    df_match_events = df_match_events.merge(
        df_players[["nameFullLocal", "personId"]],
        left_on="personId",
        right_on="personId",
        how="left",
    )
    df_match_events = df_match_events.rename(columns={"nameFullLocal": "personName"})


    # print(f"Length of match events before goalkeeperName insert: {len(df_match_events)}")
    # display(df_match_events)
    # insert nameFullLocal as personName to df_match_events from df_players
    df_match_events = df_match_events.merge(
        df_players[["nameFullLocal", "personId"]],
        left_on="goalKeeperId",
        right_on="personId",
        suffixes=("", "_goalkeeper"),
        how="left",
    )
    # drop column personId_goalkeeper
    df_match_events = df_match_events.drop(columns=["personId_goalkeeper"])
    df_match_events = df_match_events.rename(columns={"nameFullLocal": "goalkeeperName"})
    # print("After inserting goalkeeperName")
    # display(df_match_events)
    df_all_match_events = pd.concat(
        [df_all_match_events, df_match_events], ignore_index=True
    )
    
    
    # print all cols in df_players
    # print(f"Columns in df_players: {df_players.columns.tolist()}")

    columns_to_keep_player_list = [
        # "added",
        # "deceased",
        "dob",
        "externalId",
        # "gender",
        # "historicalNames",
        "images",
        # "languageLocal",
        # "nameAbbreviated",
        "nameFamilyLatin",
        "nameFamilyLocal",
        "nameFullLatin",
        "nameFullLocal",
        "nameGivenLatin",
        "nameGivenLocal",
        "nationality",
        # "organizationId",
        "personId",
        # "representing",
        # "status",
        # "updated",
        "additionalDetails.height",
        "additionalDetails.weight",
        # "organization.id",
        # "organization.resourceType",
        "teamName",
        "entityId",
    ]

    df_players_filtered = df_players[
        [col for col in columns_to_keep_player_list if col in df_players.columns]
    ]
    # print("df_players_filtered after filtering")
    # display(df_players_filtered)

    df_all_players = pd.concat(
        [df_all_players, df_players_filtered], ignore_index=True
    )
    # print(f'" --> Length of all players collected: {len(df_all_players)}')

    # break



In [ ]:
df_all_players_bak = df_all_players.copy()
df_all_match_events_bak = df_all_match_events.copy()

In [ ]:
print(f'" --> Length of all players collected: {len(df_all_players)}')
# remove duplicates in df_all_players based on personId and entityId, but show first
duplicates = df_all_players.duplicated(subset=["entityId","personId"], keep="first")
display(df_all_players[duplicates])
# drop duplicates
df_all_players = df_all_players.drop_duplicates(subset=["entityId","personId"], keep="first")
print(f'" --> Length of all players after dropping duplicates: {len(df_all_players)}')
display(df_all_players)
    # break

# drop table events if exists
con.execute("DROP TABLE IF EXISTS match_events")
# create if not exists table match_events in duckdb
con.execute("CREATE TABLE match_events AS SELECT * FROM df_all_match_events")
print("In Database:")
display(con.execute("SELECT * FROM match_events").df())

# drop table players if exists
con.execute("DROP TABLE IF EXISTS players")
# create if not exists table players in duckdb
con.execute("CREATE TABLE players AS SELECT * FROM df_all_players")
print("In Database:")
display(con.execute("SELECT * FROM players").df())
# Remove any columns not in the keep list
# df_match_events = pd.DataFrame(setup_events, columns=columns_to_keep_event_list)
# display(df_match_events)
# print(f"Columns: {df_match_events.columns.tolist()}")